# 📸 Lab 1: The Filter Factory — Hack Your Own Photo with OpenCV!

Yesterday you learned that an image is just a grid of numbers. Today we PROVE it — by hacking those numbers directly. 😎

By the end of this lab you will have: cropped a photo with pure math, blurred it, extracted its edges (running a real CNN-style filter **by hand**), detected faces, and built your own Instagram filter. All on YOUR photo.

## 📚 What You'll Learn
- Manipulating images as NumPy arrays 🔢
- Classic OpenCV operations: grayscale, blur, edge detection
- **Convolution filters — the heart of CNNs — applied manually** 🫀
- Face detection 👤

📦 **Dataset:** your own photo — ideally one **with faces in it** (a group photo works great for the face detection part!). Name it `my_photo.jpg`.

In [ ]:
!pip install opencv-python matplotlib numpy scikit-learn

### 🛠️ Import Libraries

In [ ]:
!pip install opencv-python -q

import cv2
import numpy as np
import matplotlib.pyplot as plt

def show(img, title='', gray=False):
    """Helper to display an image nicely."""
    plt.figure(figsize=(7, 5))
    if gray:
        plt.imshow(img, cmap='gray')
    else:
        plt.imshow(img)
    plt.axis('off')
    plt.title(title)
    plt.show()

print(f"📸 OpenCV version {cv2.__version__} loaded!")

### 📂 Load Your Photo

Upload a photo named `my_photo.jpg` next to this notebook. *(In Colab: 📁 icon on the left → upload.)* No photo? A backup image loads automatically.

⚠️ **OpenCV quirk alert:** `cv2.imread` loads colors in **BGR** order instead of RGB (a historical accident 😅). We convert immediately — remember this, it's the #1 OpenCV beginner bug!

In [ ]:
img = cv2.imread('my_photo.jpg')

if img is None:
    from sklearn.datasets import load_sample_image
    img = load_sample_image('china.jpg')[:, :, ::-1].copy()  # backup image (converted to BGR like imread)
    print("🏯 No photo found — using the backup image.")
else:
    print("📸 Your photo loaded!")

img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)   # fix the BGR quirk

show(img, 'Your canvas 🖼️')
print(f"📏 Shape: {img.shape} → (height, width, 3 channels)")

### 🔪 Section 1: Crop & Flip — With Pure Array Math!

No special functions needed. Since the image IS a NumPy array, slicing the array = cropping the image. `img[100:300, 50:400]` means *rows 100–300, columns 50–400*.

In [ ]:
h, w, _ = img.shape

# Crop the center of the image — pure slicing! 🔪
center_crop = img[h//4 : 3*h//4, w//4 : 3*w//4]
show(center_crop, 'Center crop — just array slicing!')

# Flip horizontally: reverse the columns with ::-1 🪞
mirror = img[:, ::-1]
show(mirror, 'Mirror — the columns, reversed!')

### 🎯 Mini Exercise 1.1
Flip the image **upside down**. (Hint: the mirror reversed the *columns*... which dimension do you reverse now? 🙃)

In [ ]:
# TO-DO: create and show the upside-down version


### 🎨 Section 2: Grayscale & Blur

**Grayscale** collapses the 3 color channels into 1 brightness channel — 3× less data, and for many tasks (like edge detection) color doesn't matter.

**Gaussian blur** replaces each pixel with a weighted average of its neighbors. More neighbors = blurrier. It's used everywhere: noise removal, background effects, privacy blurring.

In [ ]:
gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
show(gray, 'Grayscale — 1 channel instead of 3', gray=True)
print(f"📏 Color shape: {img.shape}  →  Gray shape: {gray.shape}")

In [ ]:
blur_soft  = cv2.GaussianBlur(img, (15, 15), 0)
blur_heavy = cv2.GaussianBlur(img, (51, 51), 0)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, im, title in zip(axes, [img, blur_soft, blur_heavy], ['Original', 'Soft blur (15×15)', 'Heavy blur (51×51)']):
    ax.imshow(im); ax.axis('off'); ax.set_title(title)
plt.show()

### 🫀 Section 3: THE Moment — Run a CNN Filter by Hand

Remember from theory: a CNN's convolution layer slides a tiny grid of weights (a **filter/kernel**) across the image. Let's do EXACTLY that, manually, with a famous 3×3 edge-detection filter:

```
 -1  -1  -1
 -1   8  -1
 -1  -1  -1
```

Read it like this: *"compare each pixel (×8) against its 8 neighbors (×−1 each). In flat areas everything cancels to zero (black). At an EDGE, the difference explodes (bright)."* ⚡

In [ ]:
edge_kernel = np.array([[-1, -1, -1],
                        [-1,  8, -1],
                        [-1, -1, -1]])

edges_manual = cv2.filter2D(gray, -1, edge_kernel)
show(edges_manual, '⚡ Edges — found by a 3×3 filter, just like a CNN layer!', gray=True)

In [ ]:
# Professional-grade version: the Canny edge detector (blur + gradients + smart thresholding)
edges_canny = cv2.Canny(gray, 100, 200)
show(edges_canny, '⚡ Canny edges — the industry classic', gray=True)

### 🎯 Mini Exercise 3.1 — Design Your Own Filter 🧪
Change the kernel below and observe the effect! Try:
- **Sharpen:** `[[0,-1,0], [-1,5,-1], [0,-1,0]]`
- **Vertical edges only:** `[[-1,0,1], [-2,0,2], [-1,0,1]]` *(this one is called the Sobel filter)*
- Invent your own 3×3 and see what happens! 😈

In [ ]:
my_kernel = np.array([[0, -1,  0],
                      [-1, 5, -1],
                      [0, -1,  0]])   # TO-DO: experiment!

result = cv2.filter2D(img, -1, my_kernel)
show(result, 'My custom filter 🧪')

### 👤 Section 4: Face Detection

OpenCV ships with a pretrained **Haar cascade** face detector — a classic pre-deep-learning method that scans the image at multiple scales looking for face-like brightness patterns. It's what powered digital cameras' face boxes for years! 📷

In [ ]:
face_detector = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

faces = face_detector.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)
print(f"👤 Faces found: {len(faces)}")

img_faces = img.copy()
for (x, y, fw, fh) in faces:
    cv2.rectangle(img_faces, (x, y), (x + fw, y + fh), (0, 255, 0), 3)

show(img_faces, f'👤 {len(faces)} face(s) detected!')

# 0 faces? Try a clearer front-facing photo, or lower minNeighbors to 3.
# Extra faces in bushes? That's a FALSE POSITIVE — remember Day 1! 😄

### 🎯 Mini Exercise 4.1 — Privacy Mode 🕶️
Combine two things you learned: for each detected face, **blur that region** of the image. (Hint: slice the face region `img[y:y+fh, x:x+fw]`, blur the slice, and put it back in place.) This is literally how TV blurs faces!

In [ ]:
img_private = img.copy()

for (x, y, fw, fh) in faces:
    face_region = img_private[y:y+fh, x:x+fw]
    # TO-DO: blur face_region with cv2.GaussianBlur and assign it back into img_private

show(img_private, '🕶️ Privacy mode')

### 🌅 Section 5: Build Your Own Instagram Filter!

Real filters are just chains of the operations you now know. Here's a warm "vintage" filter: a **sepia** color transformation + a **vignette** (dark corners).

In [ ]:
def vintage_filter(image):
    # 1) Sepia: mix the R, G, B channels with a warm-tone matrix 🟤
    sepia_matrix = np.array([[0.393, 0.769, 0.189],
                             [0.349, 0.686, 0.168],
                             [0.272, 0.534, 0.131]])
    sepia = image @ sepia_matrix.T
    sepia = np.clip(sepia, 0, 255).astype(np.uint8)

    # 2) Vignette: multiply by a mask that fades toward the corners 🌑
    hh, ww = sepia.shape[:2]
    kernel_x = cv2.getGaussianKernel(ww, ww / 2)
    kernel_y = cv2.getGaussianKernel(hh, hh / 2)
    mask = kernel_y @ kernel_x.T
    mask = mask / mask.max()
    vintage = (sepia * mask[:, :, np.newaxis]).astype(np.uint8)
    return vintage

show(vintage_filter(img), '🌅 Your vintage filter — ready for Instagram!')

### 🎯 Mini Exercise 5.1 — Filter Designer 🎨
Create YOUR own signature filter by combining anything from today: blur + edges overlay? Grayscale with one color channel kept? Mirror + vignette? Give it a name and show it off!

In [ ]:
# TO-DO: build your signature filter and show the result!


## The GOLDEN Question 🏆

In Section 3, **YOU chose** the edge-detection kernel values (−1s and an 8) and slid them over the image.

**A CNN does the exact same sliding operation — so what is the ONE crucial difference between your hand-made filter and the filters inside a CNN's convolution layer? And why does that difference make CNNs so powerful?** 🤔

*Hint: where did YOUR kernel's numbers come from? Where do the CNN's come from?*